# 2.12 — Linear & Quadratic Programming

Linear programming (LP) and quadratic programming (QP) optimize simple objectives over linear rules. In this lesson, we will build the geometry from scratch with NumPy: LPs slide a straight objective until it rests on a feasible vertex, while convex QPs drop a quadratic bowl onto the feasible region and stop where active constraints balance the gradient.

## 📖 Concept walkthrough — build each idea from scratch

Before the worked examples, we build linear and quadratic programming one idea at a time. Run each cell in order and inspect the printed arrays — the goal is not to call a solver, but to see why vertices, active constraints, and multipliers certify the answer. This walkthrough is self-contained and uses a `_w` suffix so it never clashes with later examples.

In [ ]:
import numpy as np  # arrays, grids, linear algebra, and small numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any stochastic-looking examples.

### 1. Linear programming: feasible polygons and objective lines

A linear program maximizes a linear score $c^Tx$ subject to linear inequalities $Ax\le b$. In two variables, those inequalities carve out a polygon. The objective contours $c^Tx=\text{constant}$ are parallel lines; increasing the objective slides the line in direction $c$ until it touches the feasible polygon for the last time.

In [ ]:
A_w = np.array([[-1., 0.], [0., -1.], [1., 2.]])  # -x<=0, -y<=0, x+2y<=2.
b_w = np.array([0., 0., 2.])  # right-hand sides for the three walls.
c_w = np.array([1., 2.])  # objective: maximize x + 2y.
verts_w = np.array([[0., 0.], [2., 0.], [0., 1.]])  # intersections of active walls.

print("A shape:", A_w.shape, "b shape:", b_w.shape)
print("candidate vertices:\n", verts_w)

▶ What you'll see: three linear inequalities define a triangular feasible region with three candidate corners.

In [ ]:
scores_w = verts_w @ c_w  # evaluate c^T x at each vertex.

print("vertex scores:", scores_w)

assert np.allclose(scores_w, [0., 2., 2.])  # the two nonzero vertices tie.

▶ What you'll see: `(2,0)` and `(0,1)` both achieve objective value 2, so the LP has multiple optima.

In [ ]:
xs_w = np.linspace(0, 2.2, 200)  # x-grid for drawing the feasible edge.
edge_y_w = (2 - xs_w) / 2  # boundary x + 2y = 2.
plt.figure(figsize=(4.8, 3.6))
plt.fill([0, 2, 0], [0, 0, 1], color="lightsteelblue", alpha=0.7, label="feasible")
for level_w in [0.5, 1.2, 2.0]:  # objective lines x+2y=constant.
    plt.plot(xs_w, (level_w - xs_w) / 2, linestyle="--", label=f"cᵀx={level_w}")
plt.scatter(verts_w[:, 0], verts_w[:, 1], color="black")
plt.xlim(-0.1, 2.2); plt.ylim(-0.1, 1.25); plt.xlabel("x"); plt.ylabel("y")
plt.title("1: LP objective lines touch a face"); plt.legend(); plt.show()

▶ What you'll see: the highest drawn objective line lies exactly on the slanted edge, explaining why every point on that edge is optimal.

*Why it's done this way:* linear inequalities are kept as matrix rows because each row is one wall of the feasible set. Evaluating only the vertices is enough for a bounded LP because a linear function has no curvature: along any edge it changes at a constant rate, so a strict maximum cannot hide in the interior unless the objective is flat along an optimal face.

### 2. Enumerating vertices from active constraints

A vertex in two dimensions is created when two independent constraints are active as equalities. Instead of trusting a drawing, we can solve every pair of walls, keep intersections that satisfy all inequalities, and then evaluate the objective on those feasible candidates.

In [ ]:
pairs_w = [(0, 1), (0, 2), (1, 2)]  # each pair says which two walls are active.
found_w = []
for i_w, j_w in pairs_w:
    M_w = A_w[[i_w, j_w]]  # the active wall normals.
    rhs_w = b_w[[i_w, j_w]]  # the active wall values.
    x_w = np.linalg.solve(M_w, rhs_w)  # intersection of those two boundary lines.
    feasible_w = np.all(A_w @ x_w <= b_w + 1e-9)  # check every inequality, not just the active pair.
    print((i_w, j_w), "->", np.round(x_w, 3), "feasible?", feasible_w)
    if feasible_w:
        found_w.append(x_w)
found_w = np.array(found_w)

▶ What you'll see: all three pairwise intersections are feasible and match the triangle's three corners.

In [ ]:
obj_found_w = found_w @ c_w  # objective values at feasible vertices.
best_w = np.max(obj_found_w)  # optimal LP value.

print("found vertices:\n", found_w)
print("best value:", best_w, "best vertices:\n", found_w[np.isclose(obj_found_w, best_w)])

assert best_w == 2.0

▶ What you'll see: the enumeration produces a certificate: no listed vertex beats value 2.

In [ ]:
plt.figure(figsize=(4.8, 3.6))
plt.fill(verts_w[:, 0], verts_w[:, 1], color="lightsteelblue", alpha=0.65, label="feasible polygon")
plt.plot(xs_w, edge_y_w, color="seagreen", label="x+2y=2")
for level_w2 in [1.0, best_w]:
    plt.plot(xs_w, (level_w2 - xs_w) / 2, linestyle="--", label=f"cᵀx={level_w2:g}")
plt.scatter(found_w[:, 0], found_w[:, 1], color="black", zorder=3, label="enumerated vertices")
plt.scatter(found_w[np.isclose(obj_found_w, best_w), 0], found_w[np.isclose(obj_found_w, best_w), 1],
            s=120, facecolors="none", edgecolors="crimson", linewidths=2, label="optimal face endpoints")
plt.xlim(-0.1, 2.2); plt.ylim(-0.1, 1.25); plt.xlabel("x"); plt.ylabel("y")
plt.title("2: vertex enumeration certifies the LP"); plt.legend(); plt.show()

▶ What you'll see: the enumerated feasible intersections sit on the polygon, with the tied optimal endpoints circled.

*Why it's done this way:* active-set enumeration mirrors what solvers reason about. The active constraints turn inequalities into equalities, linear algebra proposes a vertex, and the final feasibility check prevents intersections outside the polygon from being mistaken for legal solutions.

### 3. Quadratic programming: projecting a point onto linear constraints

A convex QP minimizes a quadratic bowl over linear constraints. A simple example is projection: minimize $\frac12\|x-p\|^2$ subject to $a^Tx\le b$. If $p$ is infeasible, the closest feasible point lies on the active wall, and the correction is along the wall normal.

In [ ]:
p_w = np.array([2., 2.])  # unconstrained bowl center.
a_w = np.array([1., 2.])  # wall normal for x + 2y <= 2.
b_proj_w = 2.0
violation_w = float(a_w @ p_w - b_proj_w)  # positive means p is outside.

print("a^T p - b =", violation_w)

assert violation_w == 4.0

▶ What you'll see: the unconstrained minimizer violates the wall by 4, so the constraint must become active.

In [ ]:
step_w = violation_w / float(a_w @ a_w) * a_w  # orthogonal correction to the hyperplane.
proj_w = p_w - step_w  # closest feasible point on a^T x = b.

print("correction:", step_w)
print("projection:", proj_w)
print("active value a^T x:", float(a_w @ proj_w))

assert np.allclose(proj_w, [1.2, 0.4])

▶ What you'll see: the nearest point on `x + 2y = 2` is `(1.2, 0.4)`; it exactly satisfies the active constraint.

In [ ]:
plt.figure(figsize=(4.6, 3.6))
plt.fill([0, 2, 0], [0, 0, 1], color="honeydew", alpha=0.8, label="feasible half-triangle")
plt.plot(xs_w, edge_y_w, color="seagreen", label="x+2y=2")
plt.scatter([p_w[0]], [p_w[1]], color="crimson", label="unconstrained p")
plt.scatter([proj_w[0]], [proj_w[1]], color="black", label="projection")
plt.plot([p_w[0], proj_w[0]], [p_w[1], proj_w[1]], color="crimson", linestyle="--")
plt.xlim(-0.1, 2.3); plt.ylim(-0.1, 2.2); plt.xlabel("x"); plt.ylabel("y")
plt.title("3: QP projection hits an active wall"); plt.legend(); plt.show()

▶ What you'll see: the shortest correction from `p` to the feasible wall is perpendicular to the wall, not horizontal or vertical.

*Why it's done this way:* the objective is squared distance, whose level sets are circles around `p`. The first circle that touches the feasible halfspace is tangent to the boundary, so the radius to the solution must align with the constraint normal. Algebraically, subtracting `(a^T p-b)/(a^T a) a` removes exactly the normal-direction violation.

### 4. KKT multipliers: balancing objective gradient and wall normal

KKT conditions explain why the projection stopped at that wall. For minimizing $\frac12\|x-p\|^2$ with $a^Tx\le b$, the gradient is $x-p$. At the constrained solution, stationarity says $x-p+\lambda a=0$, feasibility says $a^Tx\le b$, and complementary slackness says $\lambda(a^Tx-b)=0$.

In [ ]:
lambda_w = violation_w / float(a_w @ a_w)  # multiplier for the active halfspace.
grad_w = proj_w - p_w  # gradient of 1/2||x-p||^2 at the projection.
balance_w = grad_w + lambda_w * a_w  # should be zero at a KKT point.

print("lambda:", lambda_w)
print("gradient:", grad_w)
print("grad + lambda*a:", np.round(balance_w, 12))

assert np.allclose(balance_w, [0., 0.])

▶ What you'll see: the negative objective gradient is exactly cancelled by a positive multiple of the active wall normal.

In [ ]:
slack_w = b_proj_w - float(a_w @ proj_w)  # b - a^T x for <= constraint.
comp_w = lambda_w * slack_w  # complementary slackness product.

print("slack:", slack_w, "lambda*slack:", comp_w)

assert abs(comp_w) < 1e-12

▶ What you'll see: the active wall has zero slack, so a positive multiplier is allowed.

In [ ]:
plt.figure(figsize=(4.8, 3.6))
plt.fill([0, 2, 0], [0, 0, 1], color="honeydew", alpha=0.8, label="feasible")
plt.plot(xs_w, edge_y_w, color="seagreen", label="active wall")
plt.scatter([proj_w[0]], [proj_w[1]], color="black", zorder=4, label="KKT point")
plt.arrow(proj_w[0], proj_w[1], grad_w[0], grad_w[1], head_width=0.06, length_includes_head=True,
          color="crimson", label="gradient x-p")
normal_push_w = lambda_w * a_w
plt.arrow(proj_w[0], proj_w[1], normal_push_w[0], normal_push_w[1], head_width=0.06,
          length_includes_head=True, color="royalblue", label="λa")
plt.xlim(-0.1, 2.3); plt.ylim(-0.1, 1.5); plt.xlabel("x"); plt.ylabel("y")
plt.title("4: KKT force balance at the active wall"); plt.legend(); plt.show()

▶ What you'll see: the red gradient and blue multiplier force point in opposite directions and cancel at the boundary.

*Why it's done this way:* multipliers are not arbitrary solver artifacts; they measure how hard each active constraint pushes back. A positive multiplier means relaxing the wall would improve the objective, and the stationarity equation is the vector balance between the bowl's downhill pull and the feasible wall's normal force.

### 5. Equality QP and dual certificate

Equality-constrained QPs are another clean case. To minimize $\frac12\|x\|^2$ subject to $1^Tx=1$, the smallest norm point on the line splits mass evenly. The KKT system solves the primal variables and the multiplier at once.

In [ ]:
H_w = np.eye(2)  # Hessian of 1/2||x||^2.
E_w = np.array([[1., 1.]])  # equality constraint x1 + x2 = 1.
rhs_eq_w = np.array([1.])
KKT_w = np.block([[H_w, E_w.T], [E_w, np.zeros((1, 1))]])  # [H E^T; E 0].
rhs_kkt_w = np.r_[np.zeros(2), rhs_eq_w]  # stationarity and equality right-hand side.
sol_w = np.linalg.solve(KKT_w, rhs_kkt_w)
x_eq_w, nu_w = sol_w[:2], sol_w[2]

print("x:", x_eq_w, "nu:", nu_w)

assert np.allclose(x_eq_w, [0.5, 0.5])

▶ What you'll see: symmetry produces `(0.5, 0.5)`, with a multiplier that enforces the equality.

In [ ]:
obj_eq_w = 0.5 * float(x_eq_w @ x_eq_w)  # value of 1/2||x||^2.
stationarity_eq_w = H_w @ x_eq_w + E_w.T[:, 0] * nu_w

print("objective:", obj_eq_w)
print("stationarity:", stationarity_eq_w)

assert obj_eq_w == 0.25
assert np.allclose(stationarity_eq_w, [0., 0.])

▶ What you'll see: the objective is 0.25 and the stationarity residual is zero.

In [ ]:
grid_eq_w = np.linspace(-0.2, 1.2, 121)
X_eq_w, Y_eq_w = np.meshgrid(grid_eq_w, grid_eq_w)
Z_eq_w = 0.5 * (X_eq_w ** 2 + Y_eq_w ** 2)
plt.figure(figsize=(4.4, 3.6))
plt.contour(X_eq_w, Y_eq_w, Z_eq_w, levels=10, cmap="viridis")
plt.plot(grid_eq_w, 1 - grid_eq_w, color="seagreen", label="x1+x2=1")
plt.scatter([x_eq_w[0]], [x_eq_w[1]], color="black", zorder=3, label="minimum-norm point")
plt.xlim(-0.2, 1.2); plt.ylim(-0.2, 1.2); plt.xlabel("x1"); plt.ylabel("x2")
plt.title("5: equality line touches the smallest contour"); plt.legend(); plt.show()

▶ What you'll see: circular norm contours first meet the equality line at `(0.5, 0.5)`.

*Why it's done this way:* the block KKT matrix encodes all first-order optimality equations in one linear solve. Because the Hessian is positive semidefinite (positive definite here) on the feasible directions, satisfying KKT is not just necessary; it certifies the global convex optimum.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses small
> numbers, prints every intermediate value with an inline `# ->` showing the result, draws one
> picture, and ends with an `assert`. Run them top to bottom.

### ✍️ Toy 1 · LP objectives choose a feasible face

A linear objective scores every feasible vertex. Here two vertices tie, so the whole edge between them
is optimal.

In [ ]:
import numpy as np                              # matrices and vertex scores.
import matplotlib.pyplot as plt                 # one picture per toy.

t1_rng = np.random.default_rng(0)               # -> seeded reproducibly
t1_A = np.array([[-1., 0.], [0., -1.], [1., 2.]])
t1_b = np.array([0., 0., 2.])
t1_c = np.array([1., 2.])                       # maximize x+2y
t1_vertices = np.array([[0., 0.], [2., 0.], [0., 1.]])
t1_scores = t1_vertices @ t1_c                  # objective scores       # -> [0,2,2]
t1_best = float(np.max(t1_scores))              # LP value               # -> 2.0
t1_tight = np.isclose(t1_scores, t1_best)       # optimal vertices       # -> [False,True,True]

print("A:\n", t1_A)                               # -> rows for -x, -y, x+2y
print("vertices:", t1_vertices.tolist())                    # -> [[0.0, 0.0], [2.0, 0.0], [0.0, 1.0]]
print("scores:", t1_scores.tolist())                        # -> [0.0, 2.0, 2.0]
print("optimal vertex mask:", t1_tight.tolist())            # -> [False, True, True]

assert t1_best == 2.0 and np.sum(t1_tight) == 2

plt.figure(figsize=(4.6, 3.4))
plt.fill(t1_vertices[:, 0], t1_vertices[:, 1], color="#c6dbef", alpha=0.8, label="feasible triangle")
for t1_level in [0.5, 1.2, 2.0]:
    t1_xs = np.linspace(0.0, 2.2, 6)
    plt.plot(t1_xs, (t1_level - t1_xs) / 2.0, linestyle="--", label=f"cᵀx={t1_level:g}")
plt.scatter(t1_vertices[:, 0], t1_vertices[:, 1], color="black")
plt.scatter(t1_vertices[t1_tight, 0], t1_vertices[t1_tight, 1], facecolors="none", edgecolors="crimson", s=120, linewidths=2)
plt.xlim(-0.1, 2.2)
plt.ylim(-0.1, 1.2)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Toy 1 · LP ties along a face")
plt.legend()
plt.show()

▶ What you'll see: the highest objective line rests on the slanted edge, touching two optimal vertices.

### ✍️ Toy 2 · Vertex enumeration solves active wall pairs

In two dimensions, each vertex comes from two active walls. Solve every wall pair, then keep only the
intersections satisfying all inequalities.

In [ ]:
import numpy as np                              # linear solves for wall pairs.

t2_rng = np.random.default_rng(0)               # -> seeded reproducibly
t2_A = np.array([[-1., 0.], [0., -1.], [1., 2.]])
t2_b = np.array([0., 0., 2.])
t2_c = np.array([1., 2.])
t2_pairs = [(0, 1), (0, 2), (1, 2)]
t2_found = []
t2_ok = []
for t2_i, t2_j in t2_pairs:
    t2_x = np.linalg.solve(t2_A[[t2_i, t2_j]], t2_b[[t2_i, t2_j]])
    t2_feasible = bool(np.all(t2_A @ t2_x <= t2_b + 1e-9))
    t2_found.append(t2_x)
    t2_ok.append(t2_feasible)
t2_found = np.array(t2_found)                   # intersections          # -> [[-0,-0],[-0,1],[2,-0]]
t2_scores = t2_found @ t2_c                     # vertex scores          # -> [0,2,2]
t2_best = float(np.max(t2_scores))              # best vertex score      # -> 2.0

print("wall pairs:", t2_pairs)                             # -> [(0, 1), (0, 2), (1, 2)]
print("intersections:", np.round(t2_found, 3).tolist())     # -> [[-0.0, -0.0], [-0.0, 1.0], [2.0, -0.0]]
print("feasible?:", t2_ok)                                  # -> [True, True, True]
print("scores:", t2_scores.tolist())                       # -> [0.0, 2.0, 2.0]

assert all(t2_ok) and t2_best == 2.0

plt.figure(figsize=(4.6, 3.4))
plt.fill([0, 2, 0], [0, 0, 1], color="#c6dbef", alpha=0.8)
plt.scatter(t2_found[:, 0], t2_found[:, 1], color="black", zorder=3, label="enumerated vertices")
plt.scatter(t2_found[np.isclose(t2_scores, t2_best), 0], t2_found[np.isclose(t2_scores, t2_best), 1], facecolors="none", edgecolors="crimson", s=120, linewidths=2, label="best")
plt.xlim(-0.1, 2.2)
plt.ylim(-0.1, 1.2)
plt.title("Toy 2 · active pairs make vertices")
plt.legend()
plt.show()

▶ What you'll see: all three pairwise intersections are legal, and the two nonzero vertices tie for best score.

### ✍️ Toy 3 · An LP ray certifies unboundedness

If a direction keeps all inequalities feasible and improves the objective, the LP can run forever along
that ray.

In [ ]:
import numpy as np                              # ray certificate arithmetic.

t3_rng = np.random.default_rng(0)               # -> seeded reproducibly
t3_A = np.array([[0., -1.]])                    # y>=0 written -y<=0
t3_c = np.array([1., 0.])                       # maximize x
t3_d = np.array([1., 0.])                       # move in +x direction
t3_Ad = t3_A @ t3_d                             # feasibility direction  # -> [0]
t3_cd = float(t3_c @ t3_d)                      # objective direction    # -> 1.0
t3_ray = np.array([0., 1., 2., 3., 4., 5.])
t3_values = t3_ray * t3_cd                      # objective along ray    # -> [0,1,2,3,4,5]

print("A d:", t3_Ad.tolist())                              # -> [0.0]
print("c^T d:", t3_cd)                                     # -> 1.0
print("ray values:", t3_values.tolist())                   # -> [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]

assert np.all(t3_Ad <= 0.0) and t3_cd > 0.0

plt.figure(figsize=(4.4, 2.8))
plt.plot(t3_ray, t3_values, marker="o", color="crimson")
plt.xlabel("ray length t")
plt.ylabel("cᵀ(td)")
plt.title("Toy 3 · objective grows forever")
plt.show()

▶ What you'll see: the objective grows linearly with no upper stopping point.

### ✍️ Toy 4 · QP projection removes normal-direction violation

Projecting a point onto `aᵀx≤b` subtracts exactly the violated amount along the wall normal.

In [ ]:
import numpy as np                              # halfspace projection.

t4_rng = np.random.default_rng(0)               # -> seeded reproducibly
t4_p = np.array([2., 2.])                       # unconstrained center   # -> [2,2]
t4_a = np.array([1., 2.])                       # wall normal            # -> [1,2]
t4_b = 2.0                                      # wall RHS               # -> 2.0
t4_violation = float(t4_a @ t4_p - t4_b)        # aᵀp-b                  # -> 4.0
t4_denom = float(t4_a @ t4_a)                   # ||a||²                 # -> 5.0
t4_step = (t4_violation / t4_denom) * t4_a      # normal correction      # -> [0.8,1.6]
t4_proj = t4_p - t4_step                        # closest feasible point # -> [1.2,0.4]
t4_active = float(t4_a @ t4_proj)               # active wall value      # -> 2.0

print("violation:", t4_violation)                          # -> 4.0
print("denominator:", t4_denom)                            # -> 5.0
print("correction:", t4_step.tolist())                     # -> [0.8, 1.6]
print("projection:", np.round(t4_proj, 3).tolist())        # -> [1.2, 0.4]
print("a^T projection:", round(t4_active, 6))              # -> 2.0

assert np.allclose(t4_proj, [1.2, 0.4]) and round(t4_active, 6) == 2.0

plt.figure(figsize=(4.6, 3.4))
t4_xs = np.linspace(0.0, 2.1, 6)
plt.plot(t4_xs, (t4_b - t4_xs) / 2.0, color="#54a24b", label="wall")
plt.scatter([t4_p[0]], [t4_p[1]], color="crimson", label="center p")
plt.scatter([t4_proj[0]], [t4_proj[1]], color="black", label="projection")
plt.plot([t4_p[0], t4_proj[0]], [t4_p[1], t4_proj[1]], linestyle="--", color="crimson")
plt.xlim(0.0, 2.2)
plt.ylim(0.0, 2.2)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Toy 4 · orthogonal projection")
plt.legend()
plt.show()

▶ What you'll see: the red center drops along the normal to the black feasible projection.

### ✍️ Toy 5 · QP multipliers balance the projection gradient

At the projected point, `x-p` is the objective gradient and `λa` is the active wall's push. They cancel.

In [ ]:
import numpy as np                              # KKT residuals.

t5_rng = np.random.default_rng(0)               # -> seeded reproducibly
t5_p = np.array([2., 2.])
t5_a = np.array([1., 2.])
t5_b = 2.0
t5_proj = np.array([1.2, 0.4])
t5_lambda = (float(t5_a @ t5_p) - t5_b) / float(t5_a @ t5_a)  # -> 0.8
t5_grad = t5_proj - t5_p                       # gradient x-p           # -> [-0.8,-1.6]
t5_push = t5_lambda * t5_a                     # multiplier force       # -> [0.8,1.6]
t5_balance = t5_grad + t5_push                 # stationarity residual  # -> [0,0]
t5_slack = t5_b - float(t5_a @ t5_proj)        # slack b-aᵀx            # -> 0.0
t5_comp = t5_lambda * t5_slack                 # complementary product  # -> 0.0

print("lambda:", t5_lambda)                                # -> 0.8
print("gradient:", t5_grad.tolist())                       # -> [-0.8, -1.6]
print("lambda*a:", t5_push.tolist())                       # -> [0.8, 1.6]
print("balance:", t5_balance.tolist())                     # -> [0.0, 0.0]
print("slack, lambda*slack:", round(t5_slack, 12), round(t5_comp, 12))  # -> 0.0 0.0

assert np.allclose(t5_balance, [0.0, 0.0]) and abs(t5_comp) < 1e-12

plt.figure(figsize=(4.6, 3.4))
t5_xs = np.linspace(0.0, 2.1, 6)
plt.plot(t5_xs, (t5_b - t5_xs) / 2.0, color="#54a24b", label="active wall")
plt.scatter([t5_proj[0]], [t5_proj[1]], color="black", zorder=3, label="KKT point")
plt.arrow(t5_proj[0], t5_proj[1], t5_grad[0], t5_grad[1], color="crimson", head_width=0.05, length_includes_head=True, label="x-p")
plt.arrow(t5_proj[0], t5_proj[1], t5_push[0], t5_push[1], color="#4c78a8", head_width=0.05, length_includes_head=True, label="λa")
plt.xlim(0.0, 2.2)
plt.ylim(0.0, 1.5)
plt.title("Toy 5 · KKT forces cancel")
plt.legend()
plt.show()

▶ What you'll see: the red gradient and blue wall force are equal and opposite.

### ✍️ Toy 6 · Equality QPs solve one block KKT system

A minimum-norm equality problem combines stationarity and feasibility in a single linear system. The
solution splits mass evenly.

In [ ]:
import numpy as np                              # block linear algebra.

t6_rng = np.random.default_rng(0)               # -> seeded reproducibly
t6_H = np.eye(2)                                # Hessian                # -> [[1,0],[0,1]]
t6_E = np.array([[1., 1.]])                     # equality row           # -> [[1,1]]
t6_rhs_eq = np.array([1.0])                     # equality RHS           # -> [1]
t6_KKT = np.block([[t6_H, t6_E.T], [t6_E, np.zeros((1, 1))]])
t6_rhs = np.r_[np.zeros(2), t6_rhs_eq]
t6_sol = np.linalg.solve(t6_KKT, t6_rhs)        # [x1,x2,nu]             # -> [0.5,0.5,-0.5]
t6_x = t6_sol[:2]                               # primal solution        # -> [0.5,0.5]
t6_nu = float(t6_sol[2])                        # equality multiplier    # -> -0.5
t6_obj = 0.5 * float(t6_x @ t6_x)               # objective              # -> 0.25
t6_stationarity = t6_H @ t6_x + t6_E.T[:, 0] * t6_nu  # residual          # -> [0,0]

print("KKT matrix:\n", t6_KKT)                     # -> [[1,0,1],[0,1,1],[1,1,0]]
print("solution x, nu:", t6_x.tolist(), t6_nu)              # -> [0.5, 0.5] -0.5
print("objective:", t6_obj)                                # -> 0.25
print("stationarity:", t6_stationarity.tolist())           # -> [0.0, 0.0]

assert np.allclose(t6_x, [0.5, 0.5]) and t6_obj == 0.25 and np.allclose(t6_stationarity, [0.0, 0.0])

plt.figure(figsize=(4, 3.4))
t6_t = np.linspace(-0.2, 1.2, 6)
plt.plot(t6_t, 1.0 - t6_t, color="#54a24b", label="x1+x2=1")
plt.scatter([t6_x[0]], [t6_x[1]], color="black", zorder=3, label="minimum norm")
plt.axis("equal")
plt.xlim(-0.2, 1.2)
plt.ylim(-0.2, 1.2)
plt.title("Toy 6 · equality KKT solve")
plt.legend()
plt.show()

▶ What you'll see: the equality line is first touched by the smallest norm point `(0.5, 0.5)`.

### ✍️ Toy 7 · Hessian eigenvalues decide QP curvature

Convex QPs need nonnegative curvature in every direction. A negative Hessian eigenvalue creates a saddle
direction.

In [ ]:
import numpy as np                              # eigenvalues and quadratic forms.

t7_rng = np.random.default_rng(0)               # -> seeded reproducibly
t7_H_good = np.array([[2., 0.], [0., 1.]])
t7_H_bad = np.array([[1., 0.], [0., -1.]])
t7_good_eigs = np.linalg.eigvalsh(t7_H_good)    # convex eigenvalues     # -> [1,2]
t7_bad_eigs = np.linalg.eigvalsh(t7_H_bad)      # indefinite eigenvalues # -> [-1,1]
t7_dirs = np.array([[1., 0.], [0., 1.], [1., 1.], [-1., 1.], [2., 0.], [0., 2.]])
t7_good_curv = np.einsum("ij,jk,ik->i", t7_dirs, t7_H_good, t7_dirs)  # -> [2,1,3,3,8,4]
t7_bad_curv = np.einsum("ij,jk,ik->i", t7_dirs, t7_H_bad, t7_dirs)    # -> [1,-1,0,0,4,-4]

print("good eigenvalues:", t7_good_eigs.tolist())           # -> [1.0, 2.0]
print("bad eigenvalues:", t7_bad_eigs.tolist())             # -> [-1.0, 1.0]
print("good directional curvature:", t7_good_curv.tolist()) # -> [2.0, 1.0, 3.0, 3.0, 8.0, 4.0]
print("bad directional curvature:", t7_bad_curv.tolist())   # -> [1.0, -1.0, 0.0, 0.0, 4.0, -4.0]

assert np.all(t7_good_eigs >= 0.0) and np.any(t7_bad_eigs < 0.0)

plt.figure(figsize=(5.2, 2.8))
t7_grid = np.linspace(-2.0, 2.0, 40)
t7_X, t7_Y = np.meshgrid(t7_grid, t7_grid)
t7_Z_good = 0.5 * (2.0 * t7_X ** 2 + t7_Y ** 2)
t7_Z_bad = 0.5 * (t7_X ** 2 - t7_Y ** 2)
plt.contour(t7_X, t7_Y, t7_Z_good, levels=8, colors="#4c78a8", alpha=0.8)
plt.contour(t7_X, t7_Y, t7_Z_bad, levels=8, colors="crimson", alpha=0.8)
plt.title("Toy 7 · bowl contours vs saddle contours")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

▶ What you'll see: the convex contours are nested bowls, while the indefinite contours bend like a saddle.

### ✍️ Toy 8 · Box-constrained QPs clip coordinates

Projecting onto a box clips each coordinate to its allowed interval. The clipped coordinates identify the
active lower and upper walls.

In [ ]:
import numpy as np                              # coordinate clipping.

t8_rng = np.random.default_rng(0)               # -> seeded reproducibly
t8_p = np.array([1.5, -0.5])                    # outside the box        # -> [1.5,-0.5]
t8_lower = np.array([0., 0.])                   # lower bounds           # -> [0,0]
t8_upper = np.array([1., 1.])                   # upper bounds           # -> [1,1]
t8_x = np.minimum(np.maximum(t8_p, t8_lower), t8_upper)  # clipped point # -> [1,0]
t8_active_lower = np.isclose(t8_x, t8_lower)    # lower active?          # -> [False,True]
t8_active_upper = np.isclose(t8_x, t8_upper)    # upper active?          # -> [True,False]
t8_correction = t8_x - t8_p                     # projection move        # -> [-0.5,0.5]

print("p:", t8_p.tolist())                                 # -> [1.5, -0.5]
print("clipped x:", t8_x.tolist())                         # -> [1.0, 0.0]
print("active lower:", t8_active_lower.tolist())           # -> [False, True]
print("active upper:", t8_active_upper.tolist())           # -> [True, False]
print("correction:", t8_correction.tolist())               # -> [-0.5, 0.5]

assert np.allclose(t8_x, [1.0, 0.0]) and t8_active_lower[1] and t8_active_upper[0]

plt.figure(figsize=(4.2, 3.2))
plt.fill([0, 1, 1, 0], [0, 0, 1, 1], color="#c6dbef", alpha=0.8, label="box")
plt.scatter([t8_p[0]], [t8_p[1]], color="crimson", label="p")
plt.scatter([t8_x[0]], [t8_x[1]], color="black", label="projection")
plt.plot([t8_p[0], t8_x[0]], [t8_p[1], t8_x[1]], linestyle="--", color="crimson")
plt.xlim(-0.2, 1.7)
plt.ylim(-0.7, 1.2)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Toy 8 · clipping reveals active walls")
plt.legend()
plt.show()

▶ What you'll see: the nearest box point is the corner `(1,0)`, with one upper and one lower wall active.

### ✍️ Toy 9 · Multipliers predict local QP sensitivity

Relaxing an active projection wall reduces the optimal value at roughly the multiplier's rate. The
linear prediction is best for small relaxations.

In [ ]:
import numpy as np                              # value sensitivity.

t9_rng = np.random.default_rng(0)               # -> seeded reproducibly
t9_p = np.array([2., 2.])
t9_a = np.array([1., 2.])
t9_b0 = 2.0
t9_lambda0 = (float(t9_a @ t9_p) - t9_b0) / float(t9_a @ t9_a)  # -> 0.8
t9_base_value = round(0.5 * float((t9_lambda0 * t9_a) @ (t9_lambda0 * t9_a)), 10)  # -> 1.6
t9_deltas = np.array([0.0, 0.05, 0.1, 0.2, 0.3, 0.4])
t9_values = 0.1 * (4.0 - t9_deltas) ** 2       # exact value after relaxing b # -> [1.6,1.5603,1.521,1.444,1.369,1.296]
t9_linear = t9_base_value - t9_lambda0 * t9_deltas  # first-order prediction    # -> [1.6,1.56,1.52,1.44,1.36,1.28]

print("lambda at b=2:", t9_lambda0)                         # -> 0.8
print("base value:", t9_base_value)                         # -> 1.6
print("true values:", np.round(t9_values, 4).tolist())      # -> [1.6, 1.5603, 1.521, 1.444, 1.369, 1.296]
print("linear prediction:", np.round(t9_linear, 4).tolist())  # -> [1.6, 1.56, 1.52, 1.44, 1.36, 1.28]

assert np.allclose(t9_values[:3], t9_linear[:3], atol=0.002)

plt.figure(figsize=(4.8, 2.8))
plt.plot(t9_deltas, t9_values, marker="o", label="true value")
plt.plot(t9_deltas, t9_linear, linestyle="--", label="v(0)-λδ")
plt.xlabel("wall relaxation δ")
plt.ylabel("optimal objective")
plt.title("Toy 9 · multiplier as sensitivity")
plt.legend()
plt.show()

▶ What you'll see: the dashed multiplier prediction tracks the true value curve closely near zero relaxation.

## 🛠️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

## 🟢 Basics (warm-up)

### Basic 1 — Write inequalities as matrix rows

**Goal.** Encode a tiny LP feasible region as `Ax <= b`, because solvers and vectorized checks work with rows of wall normals.

In [ ]:
A_b1 = np.array([[-1., 0.], [0., -1.], [1., 2.]])
b_b1 = np.array([0., 0., 2.])
points_b1 = np.array([[0., 0.], [1., 0.25], [2., 0.5]])

print("A_b1:\n", A_b1)
print("A x values:\n", points_b1 @ A_b1.T)

▶ What you'll see: each column in the printed product corresponds to one inequality tested on several points.

In [ ]:
feasible_b1 = np.all(points_b1 @ A_b1.T <= b_b1, axis=1)

print("feasible?", feasible_b1)

assert feasible_b1.tolist() == [True, True, False]

▶ What you'll see: `(2, 0.5)` fails because it violates `x + 2y <= 2`.

In [ ]:
xs_b1 = np.linspace(0, 2.2, 200)
plt.figure(figsize=(4.4, 3.2))
plt.fill([0, 2, 0], [0, 0, 1], color="lightsteelblue", alpha=0.65, label="Ax≤b feasible")
plt.plot(xs_b1, (2 - xs_b1) / 2, color="seagreen", label="x+2y=2")
plt.scatter(points_b1[feasible_b1, 0], points_b1[feasible_b1, 1], color="black", label="passes all rows")
plt.scatter(points_b1[~feasible_b1, 0], points_b1[~feasible_b1, 1], color="crimson", label="violates a row")
plt.xlim(-0.1, 2.2); plt.ylim(-0.1, 1.2); plt.xlabel("x"); plt.ylabel("y")
plt.title("Basic 1: matrix rows carve out the triangle"); plt.legend(); plt.show()

▶ What you'll see: the failing point is outside the shaded intersection of all three half-planes.

👀 Takeaway: `Ax <= b` turns many wall checks into one matrix multiplication.

### Basic 2 — Evaluate a linear objective

**Goal.** Compute $c^Tx$ for several candidates, because LP optima are found by comparing objective values over feasible candidates.

In [ ]:
c_b2 = np.array([1., 2.])
vertices_b2 = np.array([[0., 0.], [2., 0.], [0., 1.]])
scores_b2 = vertices_b2 @ c_b2

print("scores:", scores_b2)

assert np.allclose(scores_b2, [0., 2., 2.])

▶ What you'll see: two vertices tie for the best score.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["(0,0)", "(2,0)", "(0,1)"], scores_b2, color="teal")
plt.title("Basic 2: objective values at vertices")
plt.ylabel("cᵀx")
plt.show()

▶ What you'll see: the last two bars have equal height.

👀 Takeaway: a linear objective can have a whole optimal face, not just one winning point.

### Basic 3 — Check active constraints

**Goal.** Identify which inequalities are tight at a point, because active constraints define vertices and KKT conditions.

In [ ]:
A_b3 = np.array([[-1., 0.], [0., -1.], [1., 2.]])
b_b3 = np.array([0., 0., 2.])
x_b3 = np.array([0., 1.])
slack_b3 = b_b3 - A_b3 @ x_b3

print("slack:", slack_b3)

▶ What you'll see: zero slack marks constraints that hold with equality.

In [ ]:
active_b3 = np.isclose(slack_b3, 0.0)

print("active constraints:", np.where(active_b3)[0])

assert active_b3.tolist() == [True, False, True]

▶ What you'll see: at `(0,1)`, the wall `x=0` and the slanted wall are active.

In [ ]:
xs_b3 = np.linspace(0, 2.1, 200)
plt.figure(figsize=(4.4, 3.2))
plt.fill([0, 2, 0], [0, 0, 1], color="lightsteelblue", alpha=0.5, label="feasible")
plt.axvline(0, color="crimson" if active_b3[0] else "gray", linewidth=2, label="active x=0")
plt.axhline(0, color="gray", linestyle="--", label="inactive y=0")
plt.plot(xs_b3, (2 - xs_b3) / 2, color="crimson", linewidth=2, label="active x+2y=2")
plt.scatter([x_b3[0]], [x_b3[1]], color="black", zorder=4)
plt.xlim(-0.1, 2.1); plt.ylim(-0.1, 1.2); plt.xlabel("x"); plt.ylabel("y")
plt.title("Basic 3: active constraints meet at the point"); plt.legend(); plt.show()

▶ What you'll see: the point lies on the two red active walls and above the inactive bottom wall.

👀 Takeaway: active constraints are the currently binding rules; inactive constraints have positive slack.

### Basic 4 — Draw an LP feasible triangle

**Goal.** Visualize the feasible set for `x>=0`, `y>=0`, `x+2y<=2`, because LP geometry is easiest to debug as a polygon.

In [ ]:
verts_b4 = np.array([[0., 0.], [2., 0.], [0., 1.]])

print("vertices:\n", verts_b4)

area_b4 = 0.5 * 2 * 1
assert area_b4 == 1.0

▶ What you'll see: the triangle has base 2 and height 1.

In [ ]:
plt.figure(figsize=(4, 3))
plt.fill(verts_b4[:, 0], verts_b4[:, 1], color="lightsteelblue", alpha=0.8)
plt.scatter(verts_b4[:, 0], verts_b4[:, 1], color="black")
plt.xlim(-0.1, 2.1); plt.ylim(-0.1, 1.2)
plt.xlabel("x"); plt.ylabel("y"); plt.title("Basic 4: feasible triangle")
plt.show()

▶ What you'll see: the feasible region is the filled triangle below the slanted line.

👀 Takeaway: linear inequalities create flat-sided feasible regions called polytopes.

### Basic 5 — Enumerate 2D vertices from wall pairs

**Goal.** Solve pairs of active constraints, because a 2D vertex is an intersection of two independent active lines.

In [ ]:
A_b5 = np.array([[-1., 0.], [0., -1.], [1., 2.]])
b_b5 = np.array([0., 0., 2.])
pairs_b5 = [(0, 1), (0, 2), (1, 2)]
verts_b5 = []
for pair_b5 in pairs_b5:
    x_b5 = np.linalg.solve(A_b5[list(pair_b5)], b_b5[list(pair_b5)])
    if np.all(A_b5 @ x_b5 <= b_b5 + 1e-9):
        verts_b5.append(x_b5)
verts_b5 = np.array(verts_b5)

print("vertices:\n", verts_b5)

▶ What you'll see: the linear solves recover the three triangle corners.

In [ ]:
assert verts_b5.shape == (3, 2)
plt.figure(figsize=(4, 3))
plt.scatter(verts_b5[:, 0], verts_b5[:, 1], s=80, color="purple")
plt.title("Basic 5: vertices from active pairs")
plt.xlabel("x"); plt.ylabel("y"); plt.show()

▶ What you'll see: each point is a pairwise wall intersection that also passes every inequality.

👀 Takeaway: candidate vertices come from equalities; feasibility decides which candidates are legal.

### Basic 6 — Show LP degeneracy by a tied edge

**Goal.** Verify that all points on one edge can share the same objective value, because LP optima need not be unique.

In [ ]:
t_b6 = np.linspace(0, 1, 6)
edge_b6 = np.column_stack([2 * (1 - t_b6), t_b6])  # points satisfying x + 2y = 2.
c_b6 = np.array([1., 2.])
values_b6 = edge_b6 @ c_b6

print("edge points:\n", np.round(edge_b6, 2))
print("objective values:", values_b6)

▶ What you'll see: every sampled point on the edge has objective value 2.

In [ ]:
assert np.allclose(values_b6, 2.0)
plt.figure(figsize=(4, 3))
plt.plot(edge_b6[:, 0], edge_b6[:, 1], marker="o", color="seagreen")
plt.title("Basic 6: a whole optimal edge")
plt.xlabel("x"); plt.ylabel("y"); plt.show()

▶ What you'll see: the optimal set is an edge segment, not a single corner.

👀 Takeaway: a linear objective parallel to a feasible edge creates multiple optimal solutions.

### Basic 7 — Test whether a Hessian is convex

**Goal.** Check eigenvalues of a QP Hessian, because convex QP guarantees require $H\succeq0$.

In [ ]:
H_b7 = np.array([[2., 0.5], [0.5, 1.]])
eigs_b7 = np.linalg.eigvalsh(H_b7)

print("eigenvalues:", np.round(eigs_b7, 3))

assert np.all(eigs_b7 > 0)

▶ What you'll see: both eigenvalues are positive, so the quadratic bowl is convex.

In [ ]:
H_bad_b7 = np.array([[1., 0.], [0., -1.]])
eigs_bad_b7 = np.linalg.eigvalsh(H_bad_b7)

print("indefinite eigenvalues:", eigs_bad_b7)

assert np.any(eigs_bad_b7 < 0)

▶ What you'll see: one negative eigenvalue signals a saddle direction.

In [ ]:
grid_b7 = np.linspace(-2, 2, 101)
X_b7, Y_b7 = np.meshgrid(grid_b7, grid_b7)
Z_good_b7 = 0.5 * (H_b7[0, 0] * X_b7 ** 2 + 2 * H_b7[0, 1] * X_b7 * Y_b7 + H_b7[1, 1] * Y_b7 ** 2)
Z_bad_b7 = 0.5 * (H_bad_b7[0, 0] * X_b7 ** 2 + H_bad_b7[1, 1] * Y_b7 ** 2)
fig_b7, ax_b7 = plt.subplots(1, 2, figsize=(7, 3))
ax_b7[0].contour(X_b7, Y_b7, Z_good_b7, levels=12); ax_b7[0].set_title("PSD bowl")
ax_b7[1].contour(X_b7, Y_b7, Z_bad_b7, levels=12); ax_b7[1].set_title("indefinite saddle")
for axis_b7 in ax_b7:
    axis_b7.set_xlabel("x1"); axis_b7.set_ylabel("x2")
plt.suptitle("Basic 7: Hessian eigenvalues shape contours"); plt.show()

▶ What you'll see: positive eigenvalues make closed elliptical contours; a negative eigenvalue opens a saddle.

In [ ]:
eig_labels_b7 = ["H λ1", "H λ2", "bad λ1", "bad λ2"]
eig_values_b7 = np.r_[eigs_b7, eigs_bad_b7]
eig_colors_b7 = ["seagreen" if val_b7 >= 0 else "crimson" for val_b7 in eig_values_b7]
plt.figure(figsize=(4.6, 3.2))
plt.bar(eig_labels_b7, eig_values_b7, color=eig_colors_b7)
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("eigenvalue")
plt.title("Basic 7: eigenvalue signs decide convexity")
plt.show()

▶ What you'll see: the convex Hessian has only green nonnegative bars, while the indefinite Hessian has a red negative bar.

👀 Takeaway: positive semidefinite Hessians make QPs convex; negative curvature removes global guarantees.

### Basic 8 — Project onto one linear wall

**Goal.** Compute the closest point on `a^T x <= b`, because projection is the simplest convex QP with a visible active constraint.

In [ ]:
p_b8 = np.array([2., 2.])
a_b8 = np.array([1., 2.])
b_b8 = 2.0
violation_b8 = float(a_b8 @ p_b8 - b_b8)

print("violation:", violation_b8)

assert violation_b8 == 4.0

▶ What you'll see: the unconstrained point is outside the halfspace.

In [ ]:
proj_b8 = p_b8 - violation_b8 / float(a_b8 @ a_b8) * a_b8

print("projection:", proj_b8)
print("a^T projection:", float(a_b8 @ proj_b8))

assert np.allclose(proj_b8, [1.2, 0.4])

▶ What you'll see: the projection lands exactly on the boundary line.

In [ ]:
xs_b8 = np.linspace(0, 2.3, 200)
plt.figure(figsize=(4.5, 3.4))
plt.fill([0, 2, 0], [0, 0, 1], color="honeydew", alpha=0.8, label="feasible halfspace slice")
plt.plot(xs_b8, (b_b8 - xs_b8) / 2, color="seagreen", label="aᵀx=b")
plt.scatter([p_b8[0]], [p_b8[1]], color="crimson", label="p")
plt.scatter([proj_b8[0]], [proj_b8[1]], color="black", label="projection")
plt.plot([p_b8[0], proj_b8[0]], [p_b8[1], proj_b8[1]], color="crimson", linestyle="--")
plt.xlim(-0.1, 2.3); plt.ylim(-0.1, 2.2); plt.xlabel("x"); plt.ylabel("y")
plt.title("Basic 8: one-wall QP projection"); plt.legend(); plt.show()

▶ What you'll see: the shortest dashed path from `p` to the feasible set hits the active wall.

👀 Takeaway: a one-wall QP correction subtracts only the violating normal component.

### Basic 9 — Compute a KKT multiplier

**Goal.** Recover the multiplier for an active projection wall, because multipliers quantify constraint force.

In [ ]:
p_b9 = np.array([2., 2.])
a_b9 = np.array([1., 2.])
b_b9 = 2.0
lambda_b9 = (float(a_b9 @ p_b9) - b_b9) / float(a_b9 @ a_b9)

print("lambda:", lambda_b9)

assert lambda_b9 == 0.8

▶ What you'll see: the multiplier is positive because the wall is actively blocking the unconstrained minimizer.

In [ ]:
x_b9 = p_b9 - lambda_b9 * a_b9
grad_b9 = x_b9 - p_b9

print("stationarity residual:", grad_b9 + lambda_b9 * a_b9)

assert np.allclose(grad_b9 + lambda_b9 * a_b9, [0., 0.])

▶ What you'll see: the objective gradient and constraint normal cancel exactly.

In [ ]:
plt.figure(figsize=(4.5, 3.4))
plt.plot(xs_b8, (b_b9 - xs_b8) / 2, color="seagreen", label="active wall")
plt.scatter([x_b9[0]], [x_b9[1]], color="black", zorder=4, label="optimum")
plt.arrow(x_b9[0], x_b9[1], grad_b9[0], grad_b9[1], head_width=0.06, length_includes_head=True,
          color="crimson", label="gradient")
plt.arrow(x_b9[0], x_b9[1], lambda_b9 * a_b9[0], lambda_b9 * a_b9[1], head_width=0.06,
          length_includes_head=True, color="royalblue", label="λa")
plt.xlim(-0.1, 2.3); plt.ylim(-0.1, 1.5); plt.xlabel("x"); plt.ylabel("y")
plt.title("Basic 9: multiplier balances the gradient"); plt.legend(); plt.show()

▶ What you'll see: the two arrows are equal and opposite, showing stationarity as vector cancellation.

👀 Takeaway: KKT stationarity is a vector balance equation at the optimum.

### Basic 10 — Solve a minimum-norm equality QP

**Goal.** Minimize $\frac12\|x\|^2$ subject to `sum(x)=1`, because equality QPs reveal KKT block systems cleanly.

In [ ]:
H_b10 = np.eye(2)
E_b10 = np.array([[1., 1.]])
KKT_b10 = np.block([[H_b10, E_b10.T], [E_b10, np.zeros((1, 1))]])
rhs_b10 = np.array([0., 0., 1.])
sol_b10 = np.linalg.solve(KKT_b10, rhs_b10)
x_b10 = sol_b10[:2]

print("solution:", x_b10)

assert np.allclose(x_b10, [0.5, 0.5])

▶ What you'll see: the optimizer splits mass evenly between identical coordinates.

In [ ]:
obj_b10 = 0.5 * float(x_b10 @ x_b10)

print("objective:", obj_b10)

assert obj_b10 == 0.25
plt.figure(figsize=(4, 3))
plt.bar(["x1", "x2"], x_b10, color="orange")
plt.title("Basic 10: minimum-norm equality solution")
plt.ylim(0, 0.7); plt.show()

▶ What you'll see: both coordinates are equal at 0.5.

👀 Takeaway: equality-constrained convex QPs can be solved by one linear KKT system.

## 🟡 Easy

### Easy 1 — Brute-force a tiny LP grid

**Goal.** Search a dense grid before enumerating vertices, because brute force makes the geometry intuitive even though it is not how real solvers scale.

In [ ]:
x_grid_e1 = np.linspace(0, 2, 81)
y_grid_e1 = np.linspace(0, 1, 81)
X_e1, Y_e1 = np.meshgrid(x_grid_e1, y_grid_e1)
feasible_e1 = (X_e1 + 2 * Y_e1 <= 2 + 1e-12)
value_e1 = X_e1 + 2 * Y_e1
best_grid_e1 = np.max(np.where(feasible_e1, value_e1, -np.inf))

print("grid best value:", best_grid_e1)

assert abs(best_grid_e1 - 2.0) < 1e-12

▶ What you'll see: the best grid value matches the exact LP value 2.

In [ ]:
plt.figure(figsize=(4.5, 3.4))
plt.contourf(X_e1, Y_e1, np.where(feasible_e1, value_e1, np.nan), levels=12, cmap="viridis")
plt.colorbar(label="objective")
plt.title("Easy 1: objective over feasible grid")
plt.xlabel("x"); plt.ylabel("y"); plt.show()

▶ What you'll see: objective colors increase until the slanted boundary where the optimum lies.

👀 Takeaway: grid search is useful for learning, but LP structure gives exact vertex certificates.

### Easy 2 — Solve an LP by vertex enumeration

**Goal.** Enumerate vertices for a slightly larger polygon, because in 2D the LP optimum can be certified by checking feasible intersections.

In [ ]:
A_e2 = np.array([[-1., 0.], [0., -1.], [1., 1.], [2., 1.]])
b_e2 = np.array([0., 0., 4., 5.])
c_e2 = np.array([3., 2.])
verts_e2 = []
for i_e2 in range(len(b_e2)):
    for j_e2 in range(i_e2 + 1, len(b_e2)):
        M_e2 = A_e2[[i_e2, j_e2]]
        if abs(np.linalg.det(M_e2)) > 1e-9:
            x_e2 = np.linalg.solve(M_e2, b_e2[[i_e2, j_e2]])
            if np.all(A_e2 @ x_e2 <= b_e2 + 1e-9):
                verts_e2.append(x_e2)
verts_e2 = np.unique(np.round(np.array(verts_e2), 10), axis=0)

print("vertices:\n", verts_e2)

▶ What you'll see: the feasible polygon's legal corners are found from pairs of active walls.

In [ ]:
scores_e2 = verts_e2 @ c_e2
best_idx_e2 = int(np.argmax(scores_e2))

print("scores:", scores_e2)
print("best vertex:", verts_e2[best_idx_e2], "value:", scores_e2[best_idx_e2])

assert np.allclose(verts_e2[best_idx_e2], [1., 3.])

▶ What you'll see: the best objective occurs at `(1,3)` with value 9.

In [ ]:
poly_e2 = verts_e2[[0, 2, 3, 1]]
xs_e2 = np.linspace(0, 2.7, 200)
plt.figure(figsize=(4.8, 3.8))
plt.fill(poly_e2[:, 0], poly_e2[:, 1], color="lightsteelblue", alpha=0.65, label="feasible polygon")
for level_e2 in [4, 7, scores_e2[best_idx_e2]]:
    plt.plot(xs_e2, (level_e2 - c_e2[0] * xs_e2) / c_e2[1], linestyle="--", label=f"cᵀx={level_e2:g}")
plt.scatter(verts_e2[:, 0], verts_e2[:, 1], color="black", label="vertices")
plt.scatter([verts_e2[best_idx_e2, 0]], [verts_e2[best_idx_e2, 1]], s=120, color="crimson", label="best vertex")
plt.xlim(-0.1, 2.7); plt.ylim(-0.1, 4.2); plt.xlabel("x"); plt.ylabel("y")
plt.title("Easy 2: objective contours choose a vertex"); plt.legend(); plt.show()

▶ What you'll see: sliding objective lines last touch the polygon at the red `(1, 3)` vertex.

👀 Takeaway: vertex enumeration is the transparent 2D version of LP active-set reasoning.

### Easy 3 — Compare feasible and infeasible QP centers

**Goal.** Project only when needed, because a convex QP's unconstrained minimizer remains optimal if it is already feasible.

In [ ]:
centers_e3 = np.array([[0.5, 0.5], [2., 2.]])
a_e3 = np.array([1., 2.])
b_e3 = 2.0
violations_e3 = centers_e3 @ a_e3 - b_e3

print("violations:", violations_e3)

▶ What you'll see: the first center is feasible, the second violates the wall.

In [ ]:
projects_e3 = []
for p_e3, v_e3 in zip(centers_e3, violations_e3):
    if v_e3 <= 0:
        projects_e3.append(p_e3)
    else:
        projects_e3.append(p_e3 - v_e3 / float(a_e3 @ a_e3) * a_e3)
projects_e3 = np.array(projects_e3)

print("solutions:\n", projects_e3)

assert np.allclose(projects_e3[0], centers_e3[0])
assert np.allclose(projects_e3[1], [1.2, 0.4])

▶ What you'll see: feasible centers stay put; infeasible centers move orthogonally to the wall.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(centers_e3[:, 0], centers_e3[:, 1], color="crimson", label="centers")
plt.scatter(projects_e3[:, 0], projects_e3[:, 1], color="black", label="QP solutions")
plt.plot([0, 2], [1, 0], color="seagreen", label="wall")
plt.legend(); plt.title("Easy 3: projection only when infeasible"); plt.xlabel("x"); plt.ylabel("y"); plt.show()

▶ What you'll see: one point is unchanged and the other is pulled to the boundary.

👀 Takeaway: constraints matter only when they exclude the unconstrained quadratic minimizer.

### Easy 4 — Verify KKT conditions numerically

**Goal.** Compute residuals for a projection solution, because KKT checks are solver-independent optimality diagnostics.

In [ ]:
p_e4 = np.array([2., 2.])
a_e4 = np.array([1., 2.])
b_e4 = 2.0
lam_e4 = (float(a_e4 @ p_e4) - b_e4) / float(a_e4 @ a_e4)
x_e4 = p_e4 - lam_e4 * a_e4
stationarity_e4 = x_e4 - p_e4 + lam_e4 * a_e4
primal_slack_e4 = b_e4 - float(a_e4 @ x_e4)

print("x:", x_e4, "lambda:", lam_e4)
print("stationarity residual:", stationarity_e4)

▶ What you'll see: stationarity is numerically zero.

In [ ]:
dual_ok_e4 = lam_e4 >= 0
comp_e4 = lam_e4 * primal_slack_e4

print("dual feasible?", dual_ok_e4, "slack:", primal_slack_e4, "complementarity:", comp_e4)

assert dual_ok_e4 and abs(comp_e4) < 1e-12

▶ What you'll see: the multiplier is nonnegative and complementary slackness holds.

In [ ]:
xs_e4 = np.linspace(0, 2.3, 200)
plt.figure(figsize=(4.5, 3.4))
plt.fill([0, 2, 0], [0, 0, 1], color="honeydew", alpha=0.8, label="feasible")
plt.plot(xs_e4, (b_e4 - xs_e4) / 2, color="seagreen", label="active wall")
plt.scatter([p_e4[0]], [p_e4[1]], color="crimson", label="unconstrained center")
plt.scatter([x_e4[0]], [x_e4[1]], color="black", label="KKT point")
plt.plot([p_e4[0], x_e4[0]], [p_e4[1], x_e4[1]], color="crimson", linestyle="--")
plt.xlim(-0.1, 2.3); plt.ylim(-0.1, 2.2); plt.xlabel("x"); plt.ylabel("y")
plt.title("Easy 4: KKT residuals certify the projection"); plt.legend(); plt.show()

▶ What you'll see: the certified solution is exactly where the infeasible center projects onto the active constraint.

👀 Takeaway: for convex QPs, small KKT residuals are strong evidence of global optimality.

### Easy 5 — Trace a QP objective along a feasible line

**Goal.** Evaluate $\frac12\|x\|^2$ on `x1+x2=1`, because the minimum-norm equality solution is visible as a one-dimensional bowl.

In [ ]:
t_e5 = np.linspace(-0.5, 1.5, 101)
points_e5 = np.column_stack([t_e5, 1 - t_e5])
obj_e5 = 0.5 * np.sum(points_e5 ** 2, axis=1)
best_i_e5 = int(np.argmin(obj_e5))

print("best point:", points_e5[best_i_e5], "objective:", obj_e5[best_i_e5])

assert np.allclose(points_e5[best_i_e5], [0.5, 0.5])

▶ What you'll see: the minimum along the equality line occurs at equal coordinates.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(t_e5, obj_e5, color="purple")
plt.axvline(0.5, color="black", linestyle="--")
plt.title("Easy 5: QP bowl along equality")
plt.xlabel("x1, with x2=1-x1"); plt.ylabel("1/2 ||x||²"); plt.show()

▶ What you'll see: a parabola with its bottom at `x1=0.5`.

👀 Takeaway: equality constraints reduce a QP to a lower-dimensional quadratic over feasible directions.

## 🔴 Advanced

### Advanced 1 — Detect an unbounded LP direction

**Goal.** Find a feasible ray that improves the objective forever, because LPs need boundedness or certificates of unboundedness.

In [ ]:
A_a1 = np.array([[0., -1.]])  # y >= 0 written as -y <= 0.
b_a1 = np.array([0.])
c_a1 = np.array([1., 0.])  # maximize x.
d_a1 = np.array([1., 0.])  # move in +x direction.

print("A d:", A_a1 @ d_a1, "c^T d:", float(c_a1 @ d_a1))

assert np.all(A_a1 @ d_a1 <= 0) and float(c_a1 @ d_a1) > 0

▶ What you'll see: the direction preserves feasibility and increases the objective.

In [ ]:
t_a1 = np.arange(5)
values_a1 = t_a1 * float(c_a1 @ d_a1)
plt.figure(figsize=(4, 3))
plt.plot(t_a1, values_a1, marker="o", color="crimson")
plt.title("Advanced 1: objective grows along a ray")
plt.xlabel("ray length t"); plt.ylabel("cᵀ(td)"); plt.show()

▶ What you'll see: objective value grows without flattening.

👀 Takeaway: a feasible direction with `A d <= 0` and `c^T d > 0` is an LP unboundedness certificate.

### Advanced 2 — Compare convex and indefinite quadratic surfaces

**Goal.** Plot two Hessians on a grid, because QP convexity is a property of curvature in every direction.

In [ ]:
grid_a2 = np.linspace(-2, 2, 81)
X_a2, Y_a2 = np.meshgrid(grid_a2, grid_a2)
H_good_a2 = np.array([[2., 0.], [0., 1.]])
H_bad_a2 = np.array([[1., 0.], [0., -1.]])
Z_good_a2 = 0.5 * (H_good_a2[0, 0] * X_a2 ** 2 + H_good_a2[1, 1] * Y_a2 ** 2)
Z_bad_a2 = 0.5 * (H_bad_a2[0, 0] * X_a2 ** 2 + H_bad_a2[1, 1] * Y_a2 ** 2)

print("good eigs:", np.linalg.eigvalsh(H_good_a2), "bad eigs:", np.linalg.eigvalsh(H_bad_a2))

▶ What you'll see: the convex Hessian has nonnegative eigenvalues; the indefinite one has a negative eigenvalue.

In [ ]:
fig_a2, ax_a2 = plt.subplots(1, 2, figsize=(7, 3))
ax_a2[0].contour(X_a2, Y_a2, Z_good_a2, levels=12); ax_a2[0].set_title("convex bowl")
ax_a2[1].contour(X_a2, Y_a2, Z_bad_a2, levels=12); ax_a2[1].set_title("indefinite saddle")
for axis_a2 in ax_a2:
    axis_a2.set_xlabel("x"); axis_a2.set_ylabel("y")
plt.suptitle("Advanced 2: Hessian changes QP geometry"); plt.show()

▶ What you'll see: the convex contours are nested ellipses; the indefinite contours bend like a saddle.

In [ ]:
fig_a2b, ax_a2b = plt.subplots(1, 2, figsize=(7, 3))
im_good_a2 = ax_a2b[0].imshow(Z_good_a2, extent=[grid_a2.min(), grid_a2.max(), grid_a2.min(), grid_a2.max()],
                              origin="lower", cmap="viridis")
im_bad_a2 = ax_a2b[1].imshow(Z_bad_a2, extent=[grid_a2.min(), grid_a2.max(), grid_a2.min(), grid_a2.max()],
                             origin="lower", cmap="coolwarm")
ax_a2b[0].contour(X_a2, Y_a2, Z_good_a2, colors="white", linewidths=0.6, levels=8)
ax_a2b[1].contour(X_a2, Y_a2, Z_bad_a2, colors="black", linewidths=0.6, levels=8)
ax_a2b[0].set_title("convex bowl values"); ax_a2b[1].set_title("indefinite saddle values")
for axis_a2b in ax_a2b:
    axis_a2b.set_xlabel("x"); axis_a2b.set_ylabel("y")
plt.suptitle("Advanced 2: bowl versus saddle heatmaps"); plt.show()

▶ What you'll see: the convex heatmap grows outward from one center, while the saddle heatmap has opposing high and low directions.

👀 Takeaway: convex solvers rely on PSD curvature; indefinite QPs can have local behavior that is not globally reliable.

### Advanced 3 — Active-set search for a box-constrained QP

**Goal.** Minimize distance to a point over a box, because active constraints become simple clipping walls.

In [ ]:
p_a3 = np.array([1.5, -0.5])
lower_a3 = np.array([0., 0.])
upper_a3 = np.array([1., 1.])
x_a3 = np.minimum(np.maximum(p_a3, lower_a3), upper_a3)
active_lower_a3 = np.isclose(x_a3, lower_a3)
active_upper_a3 = np.isclose(x_a3, upper_a3)

print("projected x:", x_a3)
print("active lower:", active_lower_a3, "active upper:", active_upper_a3)

assert np.allclose(x_a3, [1., 0.])

▶ What you'll see: one coordinate clips to its upper wall and the other to its lower wall.

In [ ]:
plt.figure(figsize=(4, 3))
plt.fill([0, 1, 1, 0], [0, 0, 1, 1], color="lightsteelblue", alpha=0.7)
plt.scatter([p_a3[0]], [p_a3[1]], color="crimson", label="p")
plt.scatter([x_a3[0]], [x_a3[1]], color="black", label="projection")
plt.plot([p_a3[0], x_a3[0]], [p_a3[1], x_a3[1]], linestyle="--", color="crimson")
plt.xlim(-0.2, 1.7); plt.ylim(-0.7, 1.2); plt.legend(); plt.title("Advanced 3: box projection")
plt.xlabel("x1"); plt.ylabel("x2"); plt.show()

▶ What you'll see: the nearest point in the square is the clipped corner `(1,0)`.

👀 Takeaway: box QP projections reveal active constraints coordinate by coordinate.

### Advanced 4 — Solve a general equality QP with KKT

**Goal.** Minimize $\frac12x^THx+c^Tx$ subject to `Ex=f`, because real equality QPs add both curvature and a linear tilt.

In [ ]:
H_a4 = np.array([[4., 1.], [1., 2.]])
c_a4 = np.array([-1., -1.])
E_a4 = np.array([[1., 1.]])
f_a4 = np.array([1.])
KKT_a4 = np.block([[H_a4, E_a4.T], [E_a4, np.zeros((1, 1))]])
rhs_a4 = np.r_[-c_a4, f_a4]
sol_a4 = np.linalg.solve(KKT_a4, rhs_a4)
x_a4, nu_a4 = sol_a4[:2], sol_a4[2]

print("x:", np.round(x_a4, 3), "nu:", round(float(nu_a4), 3))

assert np.allclose(x_a4, [0.25, 0.75])

▶ What you'll see: curvature under the equality constraint tilts the solution away from an even split.

In [ ]:
stationarity_a4 = H_a4 @ x_a4 + c_a4 + E_a4.T[:, 0] * nu_a4
obj_a4 = 0.5 * float(x_a4 @ H_a4 @ x_a4) + float(c_a4 @ x_a4)

print("stationarity:", np.round(stationarity_a4, 12), "objective:", round(obj_a4, 3))

assert np.allclose(stationarity_a4, [0., 0.])

▶ What you'll see: the KKT residual is zero, certifying the equality-constrained optimum.

In [ ]:
grid_a4 = np.linspace(-0.2, 1.2, 121)
X_a4, Y_a4 = np.meshgrid(grid_a4, grid_a4)
Z_a4 = 0.5 * (H_a4[0, 0] * X_a4 ** 2 + 2 * H_a4[0, 1] * X_a4 * Y_a4 + H_a4[1, 1] * Y_a4 ** 2) + c_a4[0] * X_a4 + c_a4[1] * Y_a4
plt.figure(figsize=(4.6, 3.6))
plt.contour(X_a4, Y_a4, Z_a4, levels=12, cmap="viridis")
plt.plot(grid_a4, 1 - grid_a4, color="seagreen", label="x1+x2=1")
plt.scatter([x_a4[0]], [x_a4[1]], color="black", zorder=3, label="KKT solution")
plt.xlim(-0.2, 1.2); plt.ylim(-0.2, 1.2); plt.xlabel("x1"); plt.ylabel("x2")
plt.title("Advanced 4: tilted QP over an equality line"); plt.legend(); plt.show()

▶ What you'll see: tilted quadratic contours meet the equality line first at the KKT solution.

👀 Takeaway: equality QPs reduce to a symmetric block linear system when the Hessian is PSD.

### Advanced 5 — Read a dual sensitivity from a multiplier

**Goal.** Change a projection wall slightly and compare objective values, because the multiplier predicts sensitivity to constraint relaxation.

In [ ]:
p_a5 = np.array([2., 2.])
a_a5 = np.array([1., 2.])
b0_a5 = 2.0
lam0_a5 = (float(a_a5 @ p_a5) - b0_a5) / float(a_a5 @ a_a5)
def project_value_a5(bval_a5):
    viol_a5 = max(0.0, float(a_a5 @ p_a5) - bval_a5)
    x_a5 = p_a5 - viol_a5 / float(a_a5 @ a_a5) * a_a5
    return 0.5 * float((x_a5 - p_a5) @ (x_a5 - p_a5))
base_val_a5 = project_value_a5(b0_a5)

print("lambda at b=2:", lam0_a5, "objective:", base_val_a5)

assert lam0_a5 == 0.8

▶ What you'll see: the active multiplier is 0.8 at the original wall.

In [ ]:
deltas_a5 = np.array([0.0, 0.05, 0.1, 0.2])
vals_a5 = np.array([project_value_a5(b0_a5 + d_a5) for d_a5 in deltas_a5])
linear_pred_a5 = base_val_a5 - lam0_a5 * deltas_a5

print("true values:", np.round(vals_a5, 4))
print("linear sensitivity prediction:", np.round(linear_pred_a5, 4))

▶ What you'll see: small relaxations reduce the objective at roughly the multiplier's rate.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(deltas_a5, vals_a5, marker="o", label="true QP value")
plt.plot(deltas_a5, linear_pred_a5, linestyle="--", label="v(0)-λδ")
plt.title("Advanced 5: multiplier as sensitivity")
plt.xlabel("wall relaxation δ"); plt.ylabel("optimal objective"); plt.legend(); plt.show()

▶ What you'll see: the dashed first-order prediction tracks the true value closely for small relaxations.

👀 Takeaway: dual multipliers are local shadow prices for changing constraint right-hand sides.